# LeetCode 1181 — Before and After Puzzle bằng Python

Link đề: https://leetcode.com/problems/before-and-after-puzzle/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Sorting

> **Bài Premium** — đề viết theo hiểu biết về bài (và bản mô tả công khai trên kho tham khảo doocs), nên đối chiếu lại nếu có tài khoản Premium. Chữ ký hàm dưới đây suy ra từ lời giải Python công khai.

Skeleton LeetCode:

```python
class Solution:
    def beforeAndAfterPuzzles(self, phrases: List[str]) -> List[str]:
        ...
```

Ghi chú: đáp án là **danh sách chuỗi phân biệt, đã sắp xếp theo thứ tự từ điển**. Ghép cụm `phrases[i]` với `phrases[j]` (`i ≠ j`, xét **cả hai thứ tự**) khi **từ cuối** của cụm thứ nhất **trùng** **từ đầu** của cụm thứ hai; từ trùng chỉ xuất hiện **một lần** trong kết quả. Điểm cốt lõi: tách sẵn từ đầu/từ cuối, dùng **bảng băm** `từ đầu → danh sách chỉ số` để chỉ ghép những cặp khớp, rồi khử trùng bằng `set` và `sorted`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thế nào là "ghép" hai cụm, vì sao cần `i ≠ j` (theo **chỉ số**, không theo nội dung), vì sao xét cả hai thứ tự, và yêu cầu khử trùng + sắp xếp.
2. Nắm công cụ: `str.split`, `str.rsplit(maxsplit=1)`, `str.partition`, slicing chuỗi, `set`, `sorted`, `defaultdict(list)`.
3. Đi từ **thử mọi cặp và tách từ lại mỗi lần** → **tiền xử lý từ đầu/cuối** → **bảng băm theo từ đầu** → (biến thể) **nhóm theo từ nối**.
4. Xử lý chính xác cụm **một từ** (từ đầu = từ cuối) và cụm **trùng nội dung** ở hai chỉ số khác nhau.
5. Tránh lỗi: lặp từ nối hai lần, so sánh `phrases[i] != phrases[j]` thay vì `i != j`, quên khử trùng/sắp xếp, so khớp tiền tố ký tự thay vì cả từ.

## 2. Đề bài & ví dụ

Cho danh sách các **cụm từ** (phrase). Một cụm chỉ gồm chữ thường và dấu cách, không có dấu cách ở đầu/cuối, không có hai dấu cách liền nhau.

**Câu đố Trước-và-Sau** (Before and After puzzle) được tạo bằng cách **ghép** hai cụm, trong đó **từ cuối của cụm thứ nhất** giống **từ đầu của cụm thứ hai**. Khi ghép, từ chung đó chỉ được viết **một lần**.

Trả về mọi câu đố tạo được từ hai cụm `phrases[i]` và `phrases[j]` với `i != j` (**thứ tự quan trọng**, xét cả hai chiều). Kết quả là danh sách **chuỗi phân biệt**, **sắp xếp theo thứ tự từ điển**.

**Ví dụ 1:** `["writing code","code rocks"]` → `["writing code rocks"]`.

**Ví dụ 2:** `["mission statement","a quick bite to eat","a chip off the old block","chocolate bar","mission impossible","a man on a mission","block party","eat my words","bar of soap"]` → `["a chip off the old block party","a man on a mission impossible","a man on a mission statement","a quick bite to eat my words","chocolate bar of soap"]`.

**Ví dụ 3:** `["a","b","a"]` → `["a"]` (ghép `phrases[0]` với `phrases[2]`: từ cuối `"a"` = từ đầu `"a"` → `"a"`).

**Ví dụ 4:** `["ab ba","ba ab","ab ba"]` → `["ab ba ab","ba ab ba"]`.

**Ràng buộc:** `1 <= phrases.length <= 100`; `1 <= phrases[i].length <= 100`.

In [1]:
import random
import time
from collections import defaultdict
from typing import List

examples = [
    (["writing code", "code rocks"], ["writing code rocks"]),
    (["mission statement", "a quick bite to eat", "a chip off the old block", "chocolate bar",
      "mission impossible", "a man on a mission", "block party", "eat my words", "bar of soap"],
     ["a chip off the old block party", "a man on a mission impossible", "a man on a mission statement",
      "a quick bite to eat my words", "chocolate bar of soap"]),
    (["a", "b", "a"], ["a"]),
    (["ab ba", "ba ab", "ab ba"], ["ab ba ab", "ba ab ba"]),
]
for p, want in examples:
    print(p, "->", want)

['writing code', 'code rocks'] -> ['writing code rocks']
['mission statement', 'a quick bite to eat', 'a chip off the old block', 'chocolate bar', 'mission impossible', 'a man on a mission', 'block party', 'eat my words', 'bar of soap'] -> ['a chip off the old block party', 'a man on a mission impossible', 'a man on a mission statement', 'a quick bite to eat my words', 'chocolate bar of soap']
['a', 'b', 'a'] -> ['a']
['ab ba', 'ba ab', 'ab ba'] -> ['ab ba ab', 'ba ab ba']


## 3. Kiến thức nền

### 3.1 `split` — tách từ

`"a man on a mission".split()` → list các từ. Từ đầu là `[0]`, từ cuối là `[-1]`. Cụm một từ: cả hai là chính nó.

In [2]:
for p in ["a man on a mission", "code", "writing code"]:
    w = p.split()
    print(f"{p!r:22} first={w[0]!r} last={w[-1]!r}")

'a man on a mission'   first='a' last='mission'
'code'                 first='code' last='code'
'writing code'         first='writing' last='code'


### 3.2 Lấy từ đầu/cuối hiệu quả: `partition` và `rpartition`

`p.partition(" ")` trả `(trước, " ", sau)` tại dấu cách **đầu tiên**; `p.rpartition(" ")` tại dấu cách **cuối cùng**. Với cụm một từ, `partition` trả `(p, "", "")` và `rpartition` trả `("", "", p)`. Không cần tách toàn bộ cụm.

In [3]:
p = "a man on a mission"
print(p.partition(" ")[0], "|", p.rpartition(" ")[2])
print("code".partition(" "), "code".rpartition(" "))
assert p.partition(" ")[0] == "a" and p.rpartition(" ")[2] == "mission"

a | mission
('code', '', '') ('', '', 'code')


### 3.3 Ghép hai cụm bằng slicing

Cụm `A` kết thúc bằng từ `w`, cụm `B` bắt đầu bằng `w`. Kết quả = `A + B[len(w):]` — bỏ từ `w` ở đầu `B` (phần còn lại của `B` bắt đầu bằng dấu cách, hoặc rỗng nếu `B` chỉ có một từ).

In [4]:
A, B = "a man on a mission", "mission impossible"
w = B.partition(" ")[0]
print(repr(A + B[len(w):]))
print(repr("a" + "a"[len("a"):]))            # cụm một từ ghép với cụm một từ
assert A + B[len(w):] == "a man on a mission impossible"

'a man on a mission impossible'
'a'


### 3.4 `set` để khử trùng, `sorted` để sắp xếp

Nhiều cặp `(i, j)` khác nhau có thể sinh cùng một chuỗi (ví dụ 3, 4). Gom vào `set` rồi `sorted(set)` cho danh sách phân biệt đã sắp theo từ điển.

In [5]:
out = {"b c", "a b", "b c"}
print(sorted(out))

['a b', 'b c']


### 3.5 `defaultdict(list)` — bảng băm từ đầu → chỉ số

`by_first[w]` = danh sách chỉ số các cụm bắt đầu bằng từ `w`. Với cụm `i` có từ cuối `w`, mọi `j` trong `by_first[w]` (khác `i`) là một cặp ghép được.

In [6]:
by_first = defaultdict(list)
for i, p in enumerate(["code rocks", "code is fun", "writing code"]):
    by_first[p.partition(" ")[0]].append(i)
print(dict(by_first))

{'code': [0, 1], 'writing': [2]}


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 4 `["ab ba", "ba ab", "ab ba"]` (chỉ số 0, 1, 2):

| i | cụm | đầu | cuối |
|---|-----|-----|------|
| 0 | ab ba | ab | ba |
| 1 | ba ab | ba | ab |
| 2 | ab ba | ab | ba |

Cặp có `cuối(i) == đầu(j)`, `i ≠ j`:

- `(0, 1)`: `ba == ba` → `"ab ba ab"`.
- `(1, 0)`: `ab == ab` → `"ba ab ba"`. `(1, 2)`: → `"ba ab ba"` (trùng).
- `(2, 1)`: → `"ab ba ab"` (trùng).
- `(0, 2)`, `(2, 0)`: cuối `ba` ≠ đầu `ab` → không.

Khử trùng, sắp xếp: `["ab ba ab", "ba ab ba"]`.

Chú ý ví dụ 3: cụm `"a"` ở chỉ số 0 và 2 là **hai cụm khác nhau** (khác chỉ số) nên được ghép với nhau, dù nội dung giống hệt. Nhưng cụm `"b"` không thể ghép với chính nó.

In [7]:
P = ["ab ba", "ba ab", "ab ba"]
for i in range(3):
    for j in range(3):
        if i != j and P[i].split()[-1] == P[j].split()[0]:
            print((i, j), "->", P[i] + P[j][len(P[j].split()[0]):])

(0, 1) -> ab ba ab
(1, 0) -> ba ab ba
(1, 2) -> ba ab ba
(2, 1) -> ab ba ab


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi cặp có thứ tự `(i, j)`, `i ≠ j`; mỗi lần `split` cả hai cụm để lấy từ cuối/đầu, ghép bằng cách nối danh sách từ.

In [8]:
def puzzles_brute(phrases):
    res = set()
    n = len(phrases)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            a, b = phrases[i].split(), phrases[j].split()
            if a[-1] == b[0]:
                res.add(" ".join(a + b[1:]))
    return sorted(res)

for p, want in examples:
    got = puzzles_brute(p)
    print(got == want, got)
    assert got == want

True ['writing code rocks']
True ['a chip off the old block party', 'a man on a mission impossible', 'a man on a mission statement', 'a quick bite to eat my words', 'chocolate bar of soap']
True ['a']
True ['ab ba ab', 'ba ab ba']


**Độ phức tạp:** `n²` cặp × `O(L)` tách/ghép → `O(n² · L)` = `100² · 100 = 10^6` — với ràng buộc của đề, **đủ nhanh**. Cộng thêm sắp xếp kết quả `O(R log R · L)` với `R ≤ n²`.

**Nút thắt (nếu `n` lớn):** tách chuỗi lặp lại `2n` lần cho mỗi cụm, và xét cả các cặp chắc chắn không khớp.

## 6. Bước đột phá

> Hai cụm ghép được **chỉ phụ thuộc** vào (từ cuối của cụm trước, từ đầu của cụm sau). Tiền xử lý hai từ này **một lần**, rồi dùng bảng băm `từ đầu → các chỉ số` để mỗi cụm `i` chỉ duyệt các `j` **thực sự khớp**.

Invariant: `by_first[w]` chứa đúng các chỉ số `j` có `first[j] == w`. Với mỗi `i`, tập cặp hợp lệ là `{(i, j) : j ∈ by_first[last[i]], j ≠ i}`.

Quy tắc:

- Nếu `j == i` → bỏ qua (một cụm không ghép với chính nó). Lưu ý: có thể xảy ra khi cụm một từ, hoặc cụm có từ đầu = từ cuối (như `"a b a"`).
- Ngược lại → thêm `phrases[i] + phrases[j][len(last[i]):]` vào `set`.

Số cặp duyệt = số cặp khớp thật sự (cộng `n`), không còn `n²` cặp vô ích.

In [9]:
P = examples[1][0]
first = [p.partition(" ")[0] for p in P]
last = [p.rpartition(" ")[2] for p in P]
by_first = defaultdict(list)
for j, w in enumerate(first):
    by_first[w].append(j)
for i, p in enumerate(P):
    print(f"{p!r:28} last={last[i]!r:10} -> ghép với chỉ số {[j for j in by_first.get(last[i], []) if j != i]}")

'mission statement'          last='statement' -> ghép với chỉ số []
'a quick bite to eat'        last='eat'      -> ghép với chỉ số [7]
'a chip off the old block'   last='block'    -> ghép với chỉ số [6]
'chocolate bar'              last='bar'      -> ghép với chỉ số [8]
'mission impossible'         last='impossible' -> ghép với chỉ số []
'a man on a mission'         last='mission'  -> ghép với chỉ số [0, 4]
'block party'                last='party'    -> ghép với chỉ số []
'eat my words'               last='words'    -> ghép với chỉ số []
'bar of soap'                last='soap'     -> ghép với chỉ số []


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tiền xử lý từ đầu/cuối, vẫn duyệt mọi cặp

In [10]:
def puzzles_precompute(phrases):
    first = [p.partition(" ")[0] for p in phrases]
    last = [p.rpartition(" ")[2] for p in phrases]
    res = set()
    n = len(phrases)
    for i in range(n):
        for j in range(n):
            if i != j and last[i] == first[j]:
                res.add(phrases[i] + phrases[j][len(first[j]):])
    return sorted(res)

for p, want in examples:
    assert puzzles_precompute(p) == want
print("precompute OK")

precompute OK


**Độ phức tạp:** `O(n · L)` tiền xử lý + `O(n²)` so sánh từ (mỗi so sánh `O(độ dài từ)`) + `O(R · L)` ghép. Bộ nhớ `O(n · L + R · L)`.

### 7.2 Cách 3 — Bảng băm `từ đầu → chỉ số` (lời giải chuẩn)

In [11]:
def puzzles_hash(phrases):
    first = [p.partition(" ")[0] for p in phrases]
    last = [p.rpartition(" ")[2] for p in phrases]
    by_first = defaultdict(list)
    for j, w in enumerate(first):
        by_first[w].append(j)
    res = set()
    for i, p in enumerate(phrases):
        for j in by_first.get(last[i], ()):
            if j != i:
                res.add(p + phrases[j][len(first[j]):])
    return sorted(res)

for p, want in examples:
    assert puzzles_hash(p) == want
print("hash OK")

hash OK


**Độ phức tạp:** `O(n · L)` tiền xử lý + `O(n + K)` duyệt với `K` = số cặp khớp + `O(K · L)` ghép + `O(R log R · L)` sắp xếp. Khi ít cặp khớp, nhanh hơn rõ rệt so với `O(n²)`.

Dùng `by_first.get(w, ())` thay vì `by_first[w]` để **không** tạo khoá rỗng mới trong `defaultdict` khi tra cứu từ không tồn tại.

### 7.3 Cách 4 — Nhóm theo "từ nối": hai bảng băm

Nhóm cụm theo **từ cuối** (`by_last`) và theo **từ đầu** (`by_first`). Với mỗi từ nối `w` xuất hiện ở cả hai bảng, ghép mọi `i ∈ by_last[w]` với mọi `j ∈ by_first[w]`, `i ≠ j`. Cách nhìn này hữu ích khi muốn **đếm** số câu đố theo từng từ nối, hoặc song song hoá theo từ.

In [12]:
def puzzles_join_word(phrases):
    by_first, by_last = defaultdict(list), defaultdict(list)
    for idx, p in enumerate(phrases):
        by_first[p.partition(" ")[0]].append(idx)
        by_last[p.rpartition(" ")[2]].append(idx)
    res = set()
    for w in by_last.keys() & by_first.keys():
        for i in by_last[w]:
            for j in by_first[w]:
                if i != j:
                    res.add(phrases[i] + phrases[j][len(w):])
    return sorted(res)

for p, want in examples:
    assert puzzles_join_word(p) == want
print("join word OK")

join word OK


`by_last.keys() & by_first.keys()` — phép **giao** trên view khoá của dict, cho các từ vừa là từ cuối của ít nhất một cụm vừa là từ đầu của ít nhất một cụm.

**Độ phức tạp:** như cách 3.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3 `["a", "b", "a"]` và ví dụ 4.

| i | cụm | last | by_first[last] | j hợp lệ | sinh ra |
|---|-----|------|----------------|----------|---------|
| 0 | a | a | [0, 2] | 2 | "a" |
| 1 | b | b | [1] | — | — |
| 2 | a | a | [0, 2] | 0 | "a" (trùng) |

In [13]:
def trace_hash(phrases):
    first = [p.partition(" ")[0] for p in phrases]
    last = [p.rpartition(" ")[2] for p in phrases]
    by_first = defaultdict(list)
    for j, w in enumerate(first):
        by_first[w].append(j)
    print("by_first =", dict(by_first))
    res = set()
    for i, p in enumerate(phrases):
        cands = by_first.get(last[i], [])
        made = []
        for j in cands:
            if j != i:
                s = p + phrases[j][len(first[j]):]
                made.append((j, s))
                res.add(s)
        print(f"i={i} {p!r:8} last={last[i]!r:5} candidates={cands} -> {made}")
    print("kết quả:", sorted(res))
    return sorted(res)

assert trace_hash(["a", "b", "a"]) == ["a"]
print()
assert trace_hash(["ab ba", "ba ab", "ab ba"]) == ["ab ba ab", "ba ab ba"]

by_first = {'a': [0, 2], 'b': [1]}
i=0 'a'      last='a'   candidates=[0, 2] -> [(2, 'a')]
i=1 'b'      last='b'   candidates=[1] -> []
i=2 'a'      last='a'   candidates=[0, 2] -> [(0, 'a')]
kết quả: ['a']

by_first = {'ab': [0, 2], 'ba': [1]}
i=0 'ab ba'  last='ba'  candidates=[1] -> [(1, 'ab ba ab')]
i=1 'ba ab'  last='ab'  candidates=[0, 2] -> [(0, 'ba ab ba'), (2, 'ba ab ba')]
i=2 'ab ba'  last='ba'  candidates=[1] -> [(1, 'ab ba ab')]
kết quả: ['ab ba ab', 'ba ab ba']


## 9. Code nộp LeetCode

In [14]:
from collections import defaultdict
from typing import List


class Solution:
    def beforeAndAfterPuzzles(self, phrases: List[str]) -> List[str]:
        first = [p.partition(" ")[0] for p in phrases]
        last = [p.rpartition(" ")[2] for p in phrases]
        by_first = defaultdict(list)
        for j, w in enumerate(first):
            by_first[w].append(j)
        res = set()
        for i, p in enumerate(phrases):
            for j in by_first.get(last[i], ()):
                if j != i:                                   # khác CHỈ SỐ, không phải khác nội dung
                    res.add(p + phrases[j][len(first[j]):])  # từ nối chỉ viết một lần
        return sorted(res)


sol = Solution()
for p, want in examples:
    assert sol.beforeAndAfterPuzzles(p) == want
print("Solution OK")

Solution OK


## 10. Vì sao đúng

1. **Đủ cặp:** một cặp `(i, j)` hợp lệ ⟺ `i ≠ j` và `last[i] == first[j]` ⟺ `j ∈ by_first[last[i]]` và `j ≠ i`. Thuật toán duyệt mọi `i` và mọi `j` trong `by_first[last[i]]`, nên không bỏ sót và không thêm cặp sai. Cả hai thứ tự được xét vì `(i, j)` và `(j, i)` là hai lần duyệt khác nhau (với `i` khác nhau).
2. **Ghép đúng:** `phrases[j]` bắt đầu bằng `first[j]` = `last[i]`, theo sau là `" ..."` hoặc rỗng. `phrases[j][len(first[j]):]` là phần sau từ nối (giữ dấu cách đứng đầu). Nối vào `phrases[i]` (kết thúc bằng từ nối) → từ nối xuất hiện đúng một lần, khoảng trắng đúng chuẩn.
3. **Phân biệt & thứ tự:** `set` loại trùng, `sorted` sắp theo từ điển như đề yêu cầu.

## 11. Time / Space Complexity

Gọi `n` = số cụm, `L` = độ dài tối đa một cụm (≤ 100), `K` = số cặp khớp (≤ `n²`), `R` = số câu đố phân biệt (≤ `K`).

| Cách | Time | Space |
|------|------|-------|
| 1. Brute | `O(n² L + R log R · L)` | `O(R L)` |
| 2. Tiền xử lý + mọi cặp | `O(nL + n² + K L + R log R · L)` | `O(nL + R L)` |
| 3. Bảng băm từ đầu | `O(nL + n + K L + R log R · L)` | `O(nL + R L)` |
| 4. Nhóm theo từ nối | như 3 | như 3 |

Với `n = 100`, `L = 100` mọi cách đều rất nhanh; cách 3/4 tốt hơn khi `n` lớn và ít cặp khớp.

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [15]:
impls = {
    "brute": puzzles_brute,
    "precompute": puzzles_precompute,
    "hash": puzzles_hash,
    "join_word": puzzles_join_word,
    "Solution": Solution().beforeAndAfterPuzzles,
}
edge = [
    (["a"], []),                                   # một cụm, không có cặp
    (["a", "a"], ["a"]),                           # hai cụm trùng nội dung, khác chỉ số
    (["a b a"], []),                               # đầu = cuối nhưng chỉ một cụm
    (["a b a", "a b a"], ["a b a b a"]),
    (["x y", "z w"], []),                          # không khớp
    (["ab c", "a d"], []),                         # 'c' ≠ 'a'; và 'a' không phải tiền tố của từ 'ab'
    (["c ab", "a d"], []),                         # từ cuối "ab" ≠ từ đầu "a" (không so tiền tố ký tự)
    (["c a", "a d", "a"], ["a d", "c a", "c a d"]),
]
for name, f in impls.items():
    for p, want in examples + edge:
        assert f(p) == want, (name, p, f(p))
print("all impls pass", len(examples + edge))

all impls pass 12


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Từ vựng nhỏ `{a, b, ab}`, mỗi cụm 1–3 từ, thường có cụm trùng → nhiều cặp khớp và nhiều kết quả trùng.

In [16]:
def random_phrases(rng, n):
    vocab = ["a", "b", "ab"]
    return [" ".join(rng.choice(vocab) for _ in range(rng.randint(1, 3))) for _ in range(n)]

rng = random.Random(1181)
for _ in range(400):
    p = random_phrases(rng, rng.randint(1, 8))
    want = puzzles_brute(p)
    for name, f in impls.items():
        assert f(p) == want, (name, p)
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`n = 100` cụm dài ~100 ký tự; trường hợp xấu nhất khi **mọi** cụm có cùng từ đầu và từ cuối (`n² ≈ 10^4` cặp khớp).

In [17]:
rng = random.Random(3)
worst = ["x " + " ".join("".join(rng.choice("abc") for _ in range(4)) for _ in range(18)) + " x" for _ in range(100)]
mixed = [" ".join("".join(rng.choice("ab") for _ in range(rng.randint(1, 2))) for _ in range(rng.randint(1, 30))) for _ in range(100)]
for name_case, p in [("worst", worst), ("mixed", mixed)]:
    for name in ["brute", "hash", "join_word", "Solution"]:
        t0 = time.perf_counter()
        r = impls[name](p)
        print(f"{name_case:6} {name:10} {len(r)} puzzles {time.perf_counter() - t0:.3f}s")
    assert puzzles_brute(p) == Solution().beforeAndAfterPuzzles(p)

worst  brute      9900 puzzles 0.018s
worst  hash       9900 puzzles 0.007s
worst  join_word  9900 puzzles 0.005s
worst  Solution   9900 puzzles 0.005s
mixed  brute      1691 puzzles 0.009s
mixed  hash       1691 puzzles 0.001s
mixed  join_word  1691 puzzles 0.001s
mixed  Solution   1691 puzzles 0.001s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nối thẳng hai cụm (lặp từ nối)

In [18]:
def wrong_double_word(phrases):
    res = set()
    for i, a in enumerate(phrases):
        for j, b in enumerate(phrases):
            if i != j and a.split()[-1] == b.split()[0]:
                res.add(a + " " + b)
    return sorted(res)

p = ["writing code", "code rocks"]
print("sai:", wrong_double_word(p), "| đúng:", puzzles_hash(p))
assert wrong_double_word(p) != puzzles_hash(p)

sai: ['writing code code rocks'] | đúng: ['writing code rocks']


**Vì sao sai:** từ nối `"code"` xuất hiện hai lần. **Sửa:** bỏ từ đầu của cụm sau: `a + b[len(first_b):]`.

### Lỗi 2 — So sánh nội dung `a != b` thay vì chỉ số `i != j`

In [19]:
def wrong_content_neq(phrases):
    res = set()
    for a in phrases:
        for b in phrases:
            if a != b and a.rpartition(" ")[2] == b.partition(" ")[0]:
                res.add(a + b[len(b.partition(" ")[0]):])
    return sorted(res)

p = ["a", "b", "a"]
print("sai:", wrong_content_neq(p), "| đúng:", puzzles_hash(p))
assert wrong_content_neq(p) != puzzles_hash(p)

sai: [] | đúng: ['a']


**Vì sao sai:** hai cụm `"a"` ở chỉ số 0 và 2 là **hai cụm khác nhau**, được phép ghép. Điều kiện của đề là `i != j`. **Sửa:** duyệt theo chỉ số.

### Lỗi 3 — Quên khử trùng hoặc quên sắp xếp

In [20]:
def wrong_no_dedup(phrases):
    out = []
    for i, a in enumerate(phrases):
        for j, b in enumerate(phrases):
            if i != j and a.split()[-1] == b.split()[0]:
                out.append(a + b[len(b.split()[0]):])
    return out

p = ["ab ba", "ba ab", "ab ba"]
print("sai:", wrong_no_dedup(p), "| đúng:", puzzles_hash(p))
assert wrong_no_dedup(p) != puzzles_hash(p)

sai: ['ab ba ab', 'ba ab ba', 'ba ab ba', 'ab ba ab'] | đúng: ['ab ba ab', 'ba ab ba']


**Vì sao sai:** đề yêu cầu chuỗi **phân biệt** và **sắp theo từ điển**. **Sửa:** `sorted(set(...))`.

### Lỗi 4 — So khớp bằng `startswith` / `endswith` trên ký tự

In [21]:
def wrong_startswith(phrases):
    res = set()
    for i, a in enumerate(phrases):
        w = a.rpartition(" ")[2]
        for j, b in enumerate(phrases):
            if i != j and b.startswith(w):              # "ab c".startswith("a") → True!
                res.add(a + b[len(w):])
    return sorted(res)

p = ["c a", "ab d"]
print("sai:", wrong_startswith(p), "| đúng:", puzzles_hash(p))
assert wrong_startswith(p) != puzzles_hash(p)

sai: ['c ab d'] | đúng: []


**Vì sao sai:** `"ab d".startswith("a")` đúng nhưng từ đầu là `"ab"`, không phải `"a"`; nên code sai ghép ra `"c a" + "b d"` = `"c ab d"` dù hai cụm không hề khớp. **Sửa:** so sánh **nguyên từ**: `last[i] == first[j]`.

## 14. Mẹo & nhận dạng pattern

- **"Ghép hai phần tử khi khoá của A khớp khoá của B"** → bảng băm theo khoá của B, tra bằng khoá của A (tương tự join trong cơ sở dữ liệu — *hash join*).
- **`i ≠ j` theo chỉ số** khi dữ liệu có thể trùng nội dung → luôn duyệt theo chỉ số.
- **Kết quả "phân biệt, sắp xếp"** → `sorted(set(...))`.
- Lấy từ đầu/cuối: `partition(" ")[0]` / `rpartition(" ")[2]` — không cần `split` cả chuỗi.
- So khớp **cả từ**, không so tiền tố ký tự.

## 15. Biến thể

1. Trả về **số lượng** câu đố phân biệt (chỉ cần `len(set)`).
2. Cho phép ghép **chuỗi nhiều cụm** (`A → B → C`) — trở thành bài đường đi trong đồ thị (nút = cụm, cạnh khi khớp từ).
3. Khớp **k từ cuối** với **k từ đầu** — khoá băm là tuple k từ.
4. **LC 1065 / LC 336 Palindrome Pairs** — ghép cặp chuỗi theo điều kiện, dùng bảng băm.
5. **LC 2399/word chain** — kiểu "nối chữ": từ cuối/chữ cái cuối khớp đầu.

In [22]:
# Biến thể 3: khớp k từ cuối của A với k từ đầu của B (k từ chung chỉ viết một lần)
def puzzles_k(phrases, k):
    words = [p.split() for p in phrases]
    by_first = defaultdict(list)
    for j, w in enumerate(words):
        if len(w) >= k:
            by_first[tuple(w[:k])].append(j)
    res = set()
    for i, w in enumerate(words):
        if len(w) >= k:
            for j in by_first.get(tuple(w[-k:]), ()):
                if j != i:
                    res.add(" ".join(w + words[j][k:]))
    return sorted(res)

print(puzzles_k(["the big apple", "big apple pie", "apple tree"], 2))
assert puzzles_k(["the big apple", "big apple pie", "apple tree"], 2) == ["the big apple pie"]
assert puzzles_k(examples[1][0], 1) == examples[1][1]

['the big apple pie']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute | Mọi cặp, `split` mỗi lần | `O(n² L)` | `O(R L)` | `n` nhỏ (đề này), kiểm chứng |
| 2. Tiền xử lý | Từ đầu/cuối tính trước | `O(n² + K L)` | `O(nL + RL)` | Code đơn giản hơn brute |
| 3. Hash từ đầu | Chỉ duyệt cặp khớp | `O(n + K L)` | `O(nL + RL)` | **Nộp bài**, `n` lớn |
| 4. Nhóm theo từ nối | Giao hai bảng băm | `O(n + K L)` | `O(nL + RL)` | Đếm/thống kê theo từ nối |

(Chưa tính sắp xếp kết quả `O(R log R · L)`, chung cho mọi cách.)

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `beforeAndAfterPuzzles(["go home", "home run", "run fast", "go"])` = ?

**Bài 2 (tính tay):** với `["a b", "b a", "a"]`, liệt kê mọi cặp `(i, j)` khớp và kết quả.

**Bài 3 (điền TODO):** viết `merge(a, b)` ghép hai cụm (giả sử đã khớp) chỉ bằng `partition`/slicing.

**Bài 4 (tự viết lại):** viết lại cách 3 nhưng bảng băm theo **từ cuối** (`by_last`), duyệt từng `j` và tra `first[j]`.

**Bài 5 (mini challenge):** biến thể 2 — trả về độ dài (số cụm) **chuỗi ghép dài nhất** `A1 → A2 → … → Am` với các chỉ số **đôi một khác nhau** (n ≤ 10, vét cạn DFS).

In [23]:
# Bài 3 — TODO
def merge(a, b):
    # TODO: a kết thúc bằng từ w, b bắt đầu bằng w
    raise NotImplementedError

# assert merge("writing code", "code rocks") == "writing code rocks"
# assert merge("a", "a b") == "a b"

In [24]:
# Bài 5 — TODO
def longest_chain(phrases):
    # TODO: DFS trên đồ thị cạnh i -> j khi last[i] == first[j], i != j, không lặp đỉnh
    pass

# assert longest_chain(["go home", "home run", "run fast", "go"]) == 4   # go → go home → home run → run fast

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: từ đầu `go, home, run, go`; từ cuối `home, run, fast, go`. Mức 2: cặp `(0,1), (1,2), (3,0)`. Mức 3: `"go home run"`, `"home run fast"`, `"go home"`; sắp xếp.
- **Bài 2:** Mức 1: đầu `a, b, a`; cuối `b, a, a`. Mức 2: `(0,1)`, `(1,0)`, `(1,2)`, `(2,0)`. Mức 3: `"a b a"`, `"b a b"`, `"b a"`, `"a b"`.
- **Bài 3:** Mức 1: `w = b.partition(" ")[0]`. Mức 2: `b[len(w):]`. Mức 3: `a + phần đó`.
- **Bài 4:** Mức 1: `by_last[last[i]].append(i)`. Mức 2: với mỗi `j`, duyệt `by_last.get(first[j], ())`. Mức 3: bỏ `i == j`.
- **Bài 5:** Mức 1: lập danh sách kề. Mức 2: DFS với tập `visited`. Mức 3: đáp án tối thiểu là 1 (một cụm).

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().beforeAndAfterPuzzles(["go home", "home run", "run fast", "go"]) == ["go home", "go home run", "home run fast"]
# Bài 2
assert Solution().beforeAndAfterPuzzles(["a b", "b a", "a"]) == ["a b", "a b a", "b a", "b a b"]

# Bài 3
def merge_sol(a, b):
    return a + b[len(b.partition(" ")[0]):]

assert merge_sol("writing code", "code rocks") == "writing code rocks"
assert merge_sol("a", "a b") == "a b"

# Bài 4
def puzzles_by_last(phrases):
    first = [p.partition(" ")[0] for p in phrases]
    last = [p.rpartition(" ")[2] for p in phrases]
    by_last = defaultdict(list)
    for i, w in enumerate(last):
        by_last[w].append(i)
    res = set()
    for j, b in enumerate(phrases):
        for i in by_last.get(first[j], ()):
            if i != j:
                res.add(phrases[i] + b[len(first[j]):])
    return sorted(res)

rng = random.Random(81)
for _ in range(300):
    p = random_phrases(rng, rng.randint(1, 8))
    assert puzzles_by_last(p) == puzzles_brute(p)
print("bài 1-4 OK")

bài 1-4 OK


In [26]:
# Bài 5
def longest_chain_sol(phrases):
    n = len(phrases)
    first = [p.partition(" ")[0] for p in phrases]
    last = [p.rpartition(" ")[2] for p in phrases]
    adj = [[j for j in range(n) if j != i and last[i] == first[j]] for i in range(n)]
    best = 0

    def dfs(u, visited, length):
        nonlocal best
        best = max(best, length)
        for v in adj[u]:
            if v not in visited:
                visited.add(v)
                dfs(v, visited, length + 1)
                visited.remove(v)

    for s in range(n):
        dfs(s, {s}, 1)
    return best

assert longest_chain_sol(["go home", "home run", "run fast", "go"]) == 4
assert longest_chain_sol(["a", "a", "a"]) == 3
assert longest_chain_sol(["x y"]) == 1
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
first = [p.partition(' ')[0] for p in phrases]
last  = [p.rpartition(' ')[2] for p in phrases]
by_first = defaultdict(list)
for j, w in enumerate(first): by_first[w].append(j)
res = set()
for i, p in enumerate(phrases):
    for j in by_first.get(last[i], ()):
        if j != i:
            res.add(p + phrases[j][len(first[j]):])
return sorted(res)
```

**Checklist:**

- [ ] Điều kiện `i != j` theo **chỉ số**?
- [ ] Xét **cả hai thứ tự** (mỗi `i` duyệt các `j` khớp)?
- [ ] Từ nối chỉ xuất hiện **một lần** trong kết quả?
- [ ] So sánh **nguyên từ**, không dùng `startswith` ký tự?
- [ ] Cụm một từ xử lý đúng (`"a" + "a"` → `"a"`)?
- [ ] Trả `sorted(set(...))`?